<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_01_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 1: Configuration, Storage Setup, and Raw Data Ingestion

This notebook initializes the reproducible Phase 2 configuration, mounts the
persistent Google Drive storage, and prepares the Data Lake directory structure.
It checks whether Kvasir-Capsule is already available at the configured storage
location and downloads it from the official source only when it is missing.
The dataset contents are then inventoried and validated before downstream
processing begins.

### 1. Install dependencies and imports

In [1]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    gdown

In [2]:
from pathlib import Path
from collections import defaultdict

import re
import mimetypes
import os
import json
import random
import warnings
import subprocess
import shutil
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display
from collections import deque
from uuid import uuid4

import filecmp
import lzma
import stat
import tempfile

import cv2
import numpy as np
import pandas as pd
import torch
import zipfile
import tarfile
import zlib
import gzip


from tqdm.auto import tqdm

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Load declarative configuration and reproducibility

In [3]:
CONFIG = {
    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    "seed": 42,

    # --------------------------------------------------------------
    # Persistent storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Raw-dataset acquisition
    # --------------------------------------------------------------

    "dataset_download_enabled": True,

    "dataset_google_drive_folder_url": (
        "https://drive.google.com/drive/folders/"
        "18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z"
    ),

    "dataset_archive_repair_enabled": True,

    # --------------------------------------------------------------
    # Raw-dataset validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "osf_storage_subdir": (
            "osfstorage"
        ),

        "required_metadata_file": (
            "metadata.csv"
        ),

        "labelled_images_subdir": (
            "labelled_images"
        ),

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },
}


CONFIG

{'seed': 42,
 'storage_backend': 'google_drive',
 'storage_root': '/content/drive/MyDrive/MMVQA_Clinical',
 'raw_data_dir': 'data/raw/kvasir_capsule',
 'interim_data_dir': 'data/interim/phase2',
 'curated_data_dir': 'data/curated/phase2',
 'output_dir': 'outputs/phase2',
 'dataset_download_enabled': True,
 'dataset_google_drive_folder_url': 'https://drive.google.com/drive/folders/18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z',
 'dataset_archive_repair_enabled': True,
 'dataset_validation': {'osf_storage_subdir': 'osfstorage',
  'required_metadata_file': 'metadata.csv',
  'labelled_images_subdir': 'labelled_images',
  'minimum_video_files': 117,
  'minimum_labelled_images': 47238,
  'image_extensions': ['.png', '.jpg', '.jpeg'],
  'video_extensions': ['.avi', '.mp4', '.mkv']}}

In [4]:
# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

Seed set to: 42


### 3. Mount Google Drive Storage Backend

In [5]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

Mounted at /content/drive
Google Drive mounted.


### 4. Define data paths

In [6]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

{'raw_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule'),
 'interim_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2'),
 'curated_data_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2'),
 'output_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2'),
 'temporal_frames_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/interim/phase2/temporal_frames'),
 'manifests_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests'),
 'splits_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/splits'),
 'configs_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/configs'),
 'results_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results'),
 'reports_dir': PosixPath('/content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/reports'),
 'dataset_root_dir': PosixPath('/content/drive/MyDrive/MMVQA

### 5. Dataset inventory verification

In [7]:
def build_raw_dataset_inventory(
    config,
    dirs,
):
    """
    Builds a declarative inventory of the raw
    Kvasir-Capsule dataset.

    The filesystem is inspected but not modified.

    Returns:
        One dictionary containing paths, observed counts,
        expected minimums, and the final validation result.
    """

    required_dir_keys = {
        "raw_data_dir",
        "dataset_root_dir",
        "labelled_images_dir",
        "metadata_path",
    }

    missing_dir_keys = sorted(
        required_dir_keys
        - set(dirs)
    )

    if missing_dir_keys:
        raise KeyError(
            "Dataset verification is missing DIRS keys: "
            f"{missing_dir_keys}"
        )

    validation = config[
        "dataset_validation"
    ]

    required_validation_keys = {
        "minimum_video_files",
        "minimum_labelled_images",
        "image_extensions",
        "video_extensions",
    }

    missing_validation_keys = sorted(
        required_validation_keys
        - set(validation)
    )

    if missing_validation_keys:
        raise KeyError(
            "Dataset verification is missing CONFIG keys: "
            f"{missing_validation_keys}"
        )

    raw_data_dir = Path(
        dirs[
            "raw_data_dir"
        ]
    )

    dataset_root_dir = Path(
        dirs[
            "dataset_root_dir"
        ]
    )

    labelled_images_dir = Path(
        dirs[
            "labelled_images_dir"
        ]
    )

    metadata_path = Path(
        dirs[
            "metadata_path"
        ]
    )

    image_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "image_extensions"
        ]
    )

    video_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "video_extensions"
        ]
    )

    minimum_video_files = int(
        validation[
            "minimum_video_files"
        ]
    )

    minimum_labelled_images = int(
        validation[
            "minimum_labelled_images"
        ]
    )

    if minimum_video_files < 1:
        raise ValueError(
            "minimum_video_files must be positive."
        )

    if minimum_labelled_images < 1:
        raise ValueError(
            "minimum_labelled_images must be positive."
        )

    raw_data_dir_exists = (
        raw_data_dir.is_dir()
    )

    dataset_root_dir_exists = (
        dataset_root_dir.is_dir()
    )

    labelled_images_dir_exists = (
        labelled_images_dir.is_dir()
    )

    metadata_available = (
        metadata_path.is_file()
    )

    video_count = (
        sum(
            1
            for path
            in dataset_root_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in video_extensions
            )
        )
        if dataset_root_dir_exists
        else 0
    )

    labelled_image_count = (
        sum(
            1
            for path
            in labelled_images_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in image_extensions
            )
        )
        if labelled_images_dir_exists
        else 0
    )

    minimum_video_count_met = (
        video_count
        >= minimum_video_files
    )

    minimum_labelled_image_count_met = (
        labelled_image_count
        >= minimum_labelled_images
    )

    validation_passed = all(
        [
            raw_data_dir_exists,
            dataset_root_dir_exists,
            labelled_images_dir_exists,
            metadata_available,
            minimum_video_count_met,
            minimum_labelled_image_count_met,
        ]
    )

    return {
        "raw_data_dir":
            str(raw_data_dir),

        "dataset_root_dir":
            str(dataset_root_dir),

        "labelled_images_dir":
            str(labelled_images_dir),

        "metadata_path":
            str(metadata_path),

        "raw_data_dir_exists":
            raw_data_dir_exists,

        "dataset_root_dir_exists":
            dataset_root_dir_exists,

        "labelled_images_dir_exists":
            labelled_images_dir_exists,

        "metadata_available":
            metadata_available,

        "video_count":
            video_count,

        "minimum_video_files":
            minimum_video_files,

        "minimum_video_count_met":
            minimum_video_count_met,

        "labelled_image_count":
            labelled_image_count,

        "minimum_labelled_images":
            minimum_labelled_images,

        "minimum_labelled_image_count_met":
            minimum_labelled_image_count_met,

        "validation_passed":
            validation_passed,
    }


def verify_raw_dataset(
    config,
    dirs,
):
    """
    Returns True when the raw Kvasir-Capsule dataset
    satisfies all declared minimum requirements.
    """

    inventory = (
        build_raw_dataset_inventory(
            config=config,
            dirs=dirs,
        )
    )

    return bool(
        inventory[
            "validation_passed"
        ]
    )

### 6. Dataset provisioning and archive-recovery

In [8]:

# ------------------------------------------------------------------
# Expected labelled-image manifest: original thresholds retained
# ------------------------------------------------------------------

EXPECTED_LABELLED_CLASS_COUNTS = {
    "Ampulla of vater": 10,
    "Angiectasia": 866,
    "Blood - fresh": 446,
    "Blood - hematin": 12,
    "Erosion": 506,
    "Erythema": 159,
    "Foreign body": 776,
    "Ileocecal valve": 4189,
    "Lymphangiectasia": 592,
    "Normal clean mucosa": 34338,
    "Polyp": 55,
    "Pylorus": 1529,
    "Reduced mucosal view": 2906,
    "Ulcer": 854,
}

EXPECTED_LABELLED_IMAGE_COUNT = sum(
    EXPECTED_LABELLED_CLASS_COUNTS.values()
)
EXPECTED_LABELLED_CLASS_COUNT = len(EXPECTED_LABELLED_CLASS_COUNTS)

# Retained from the original class scanner.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

dataset_root_dir = Path(DIRS["dataset_root_dir"])
labelled_images_dir = Path(DIRS["labelled_images_dir"])
archive_search_root = Path(DIRS["raw_data_dir"])


# ------------------------------------------------------------------
# Dataset presence and optional download -- NEW
# ------------------------------------------------------------------

def dataset_contains_files(dataset_root):
    """
    Checks for existing files, not dataset completeness.

    A missing directory, an empty directory, or a tree containing
    only empty subdirectories is treated as absent.
    Existing files are left for the inventory and validation steps.
    """
    dataset_root = Path(dataset_root)

    if not dataset_root.exists():
        return False

    if not dataset_root.is_dir():
        raise NotADirectoryError(
            f"Dataset location is not a directory: {dataset_root}"
        )

    return any(path.is_file() for path in dataset_root.rglob("*"))


def run_gdown_download(command):
    """Streams download output and retains recent diagnostics on failure."""
    recent_output = deque(maxlen=60)

    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        try:
            if process.stdout is None:
                raise RuntimeError("Unable to capture gdown output.")

            for line in process.stdout:
                print(line, end="", flush=True)
                recent_output.append(line)

            return_code = process.wait()

        except BaseException:
            # Do not leave the download running after cell interruption.
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise

    if return_code != 0:
        diagnostics = "".join(recent_output).strip()
        raise RuntimeError(
            f"Google Drive download failed with exit code {return_code}.\n"
            f"{diagnostics or 'No diagnostic output was captured.'}\n"
            "Partially downloaded files may remain in the destination. "
            "They have not been validated as a complete dataset."
        )


def ensure_raw_dataset_present(config, dirs):
    """
    Reuses existing dataset files or downloads an absent dataset.

    Returns 'already_present' or 'downloaded'. Neither status means
    that content validation has passed.

    Existing but incomplete contents are handled by the validation
    and archive-recovery steps below, not by a full redownload.
    """
    destination = Path(dirs["dataset_root_dir"])

    if config["storage_backend"] == "google_drive":
        # Matches the /content/drive mount used by mount_storage().
        # Checking directory existence alone could accept local runtime
        # directories created while Google Drive was not mounted.
        drive_mount = Path("/content/drive")

        if not drive_mount.is_mount():
            raise RuntimeError(
                "Google Drive is not mounted at /content/drive. "
                "Run the Google Drive mount cell first."
            )

        if not destination.resolve().is_relative_to(drive_mount.resolve()):
            raise ValueError(
                "The dataset destination is outside the mounted Drive: "
                f"{destination}"
            )

    if dataset_contains_files(destination):
        print(f"Existing dataset files found at: {destination}")
        print("Skipping download; continuing with content validation.")
        return "already_present"

    if not config["dataset_download_enabled"]:
        raise RuntimeError(
            f"No dataset files were found at: {destination}. "
            "Set CONFIG['dataset_download_enabled'] = True "
            "to allow the Google Drive download."
        )

    folder_url = config["dataset_google_drive_folder_url"]
    if not isinstance(folder_url, str) or not folder_url.strip():
        raise ValueError(
            "dataset_google_drive_folder_url must be a non-empty string."
        )

    gdown_executable = shutil.which("gdown")
    if gdown_executable is None:
        raise RuntimeError(
            "gdown is not available on PATH. "
            "Run %pip install -q gdown in the dependencies cell."
        )

    destination.mkdir(parents=True, exist_ok=True)

    command = [
        gdown_executable,
        "--folder",
        folder_url.strip(),
        "-O",
        str(destination),
    ]

    print(f"Downloading Kvasir-Capsule to: {destination}")
    run_gdown_download(command)

    if not dataset_contains_files(destination):
        raise RuntimeError(
            "gdown completed but no dataset files were found at: "
            f"{destination}"
        )

    print("Download completed; continuing with content validation.")
    return "downloaded"


# ------------------------------------------------------------------
# Labelled-image scanner and report
# ------------------------------------------------------------------

def scan_labelled_image_classes(labelled_images_dir, expected_class_counts):
    """Compares the physical class inventory with the declared counts."""
    labelled_images_dir = Path(labelled_images_dir)
    actual_counts = {}

    for class_name in expected_class_counts:
        class_dir = labelled_images_dir / class_name
        actual_counts[class_name] = (
            sum(
                1
                for path in class_dir.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )
            if class_dir.is_dir()
            else 0
        )

    existing_class_dirs = (
        {
            path.name
            for path in labelled_images_dir.iterdir()
            if path.is_dir()
        }
        if labelled_images_dir.is_dir()
        else set()
    )
    expected_classes = set(expected_class_counts)
    missing_classes = [
        name for name in expected_class_counts if actual_counts[name] == 0
    ]
    incomplete_classes = {
        name: {
            "actual": actual_counts[name],
            "expected": expected,
            "missing": expected - actual_counts[name],
        }
        for name, expected in expected_class_counts.items()
        if 0 < actual_counts[name] < expected
    }
    excess_classes = {
        name: {
            "actual": actual_counts[name],
            "expected": expected,
            "excess": actual_counts[name] - expected,
        }
        for name, expected in expected_class_counts.items()
        if actual_counts[name] > expected
    }
    total_actual = sum(actual_counts.values())
    total_expected = sum(expected_class_counts.values())
    expected_classes_present = not missing_classes
    expected_class_counts_met = all(
        actual_counts[name] == expected
        for name, expected in expected_class_counts.items()
    )

    return {
        "actual_counts": actual_counts,
        "total_actual": total_actual,
        "total_expected": total_expected,
        "existing_class_count": len(existing_class_dirs & expected_classes),
        "expected_class_count": len(expected_classes),
        "missing_classes": missing_classes,
        "incomplete_classes": incomplete_classes,
        "excess_classes": excess_classes,
        "unexpected_classes": sorted(existing_class_dirs - expected_classes),
        "expected_classes_present": expected_classes_present,
        "expected_class_counts_met": expected_class_counts_met,
        "expected_total_count_met": total_actual == total_expected,
        "complete": (
            expected_classes_present
            and expected_class_counts_met
            and total_actual == total_expected
        ),
    }


def build_labelled_class_report(scan_report, expected_class_counts):
    """Builds a tabular comparison, with one row per expected class."""
    return (
        pd.Series(expected_class_counts, name="expected_images")
        .rename_axis("class_name")
        .to_frame()
        .join(
            pd.Series(scan_report["actual_counts"], name="actual_images")
            .rename_axis("class_name")
        )
        .assign(
            difference=lambda data: (
                data["actual_images"] - data["expected_images"]
            ),
            status=lambda data: (
                pd.Series("complete", index=data.index, dtype="string")
                .mask(data["actual_images"].gt(data["expected_images"]), "excess")
                .mask(data["actual_images"].lt(data["expected_images"]), "incomplete")
                .mask(data["actual_images"].eq(0), "missing")
            ),
        )
        .reset_index()
        .loc[:, [
            "class_name", "actual_images", "expected_images", "difference", "status"
        ]]
    )


# ------------------------------------------------------------------
# Archive discovery, inspection, and image recovery
# ------------------------------------------------------------------

def normalize_class_key(value):
    """Produces a comparable key for class/archive names."""
    value = re.sub(r"\.(?:tar\.gz|tgz|zip)$", "", value.lower())
    return re.sub(r"[^a-z0-9]+", "", value)


def infer_archive_class(archive_path, expected_classes):
    """Infers a class from an archive filename."""
    archive_key = normalize_class_key(Path(archive_path).name)
    for class_name in sorted(expected_classes):
        if archive_key == normalize_class_key(class_name):
            return class_name
    return None


def find_dataset_archives(search_root):
    """Finds ZIP, TAR.GZ, and TGZ archives recursively."""
    search_root = Path(search_root)
    if not search_root.is_dir():
        return []

    return sorted(
        path
        for path in search_root.rglob("*")
        if path.is_file()
        and path.name.lower().endswith((".zip", ".tar.gz", ".tgz"))
    )


def inspect_archive_for_classes(archive_path, target_classes):
    """Identifies target classes; archive errors propagate to repair."""
    archive_path = Path(archive_path)
    target_classes = set(target_classes)
    found_classes = set()

    archive_class = infer_archive_class(archive_path, target_classes)
    if archive_class is not None:
        return {archive_class}

    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            member_names = [
                member.filename for member in archive.infolist()
                if not member.is_dir()
            ]
    else:
        with tarfile.open(archive_path, "r:*") as archive:
            member_names = [
                member.name for member in archive.getmembers() if member.isfile()
            ]

    normalized_targets = {
        normalize_class_key(name): name for name in target_classes
    }
    for member_name in member_names:
        for part in Path(member_name.replace("\\", "/")).parts:
            key = normalize_class_key(part)
            if key in normalized_targets:
                found_classes.add(normalized_targets[key])

    return found_classes


def recover_required_images_from_archive(
    archive_path, labelled_images_dir, target_classes,
    raw_root, quarantine_dir, replacement_records,
):
    """
    Writes each member to a temporary file before publishing it.

    Existing files identical to the archive are reused. Files with
    different bytes are preserved in quarantine before replacement;
    a mismatch is recorded without claiming the old image is corrupt.
    Source-image decoding/QC remains a separate pipeline stage.
    """
    archive_path = Path(archive_path)
    labelled_images_dir = Path(labelled_images_dir)
    target_classes = set(target_classes)
    recovered_counts = {name: 0 for name in sorted(target_classes)}
    archive_class = infer_archive_class(archive_path, target_classes)
    destinations_seen = set()

    def image_destination(member_name):
        member_path = Path(member_name.replace("\\", "/"))
        if member_path.suffix.lower() not in IMAGE_EXTENSIONS:
            return None, None

        matched_class = archive_class
        if matched_class is None:
            matched_classes = [
                class_name for class_name in sorted(target_classes)
                if any(
                    normalize_class_key(part) == normalize_class_key(class_name)
                    for part in member_path.parts[:-1]
                )
            ]
            if len(matched_classes) > 1:
                raise ValueError(f"Ambiguous archive image class: {member_name}")
            matched_class = matched_classes[0] if matched_classes else None

        if matched_class is None:
            return None, None

        # Keep only the basename; do not extract arbitrary archive paths.
        destination = labelled_images_dir / matched_class / member_path.name
        if not destination.resolve().is_relative_to(labelled_images_dir.resolve()):
            raise ValueError(f"Image destination escapes the labelled directory: {destination}")
        if destination in destinations_seen:
            raise ValueError(f"Archive members collide at the same image path: {destination}")
        destinations_seen.add(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        return matched_class, destination

    def publish_member(source, destination, expected_size):
        with tempfile.NamedTemporaryFile(
            dir=destination.parent, prefix=".extract_", suffix=".part", delete=False
        ) as output:
            temporary_path = Path(output.name)
            try:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            except BaseException:
                output.close()
                temporary_path.unlink(missing_ok=True)
                raise

        try:
            if temporary_path.stat().st_size != expected_size:
                raise EOFError(f"Incomplete archive member for: {destination}")

            if destination.exists():
                if not destination.is_file():
                    raise IsADirectoryError(str(destination))
                if filecmp.cmp(temporary_path, destination, shallow=False):
                    return False
                quarantined = quarantine_dataset_file(
                    destination, raw_root, quarantine_dir
                )
                replacement_records.append({
                    "archive_path": str(archive_path),
                    "image_path": str(destination),
                    "quarantine_path": str(quarantined),
                    "reason": "existing_bytes_differ_from_archive",
                })

            temporary_path.replace(destination)
            return True
        finally:
            # This is only the newly created temporary extraction output.
            temporary_path.unlink(missing_ok=True)

    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or stat.S_ISLNK(member.external_attr >> 16):
                    continue
                class_name, destination = image_destination(member.filename)
                if destination is None:
                    continue
                with archive.open(member, "r") as source:
                    recovered_counts[class_name] += int(
                        publish_member(source, destination, member.file_size)
                    )
    else:
        with tarfile.open(archive_path, "r:*") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue
                class_name, destination = image_destination(member.name)
                if destination is None:
                    continue
                source = archive.extractfile(member)
                if source is None:
                    continue
                with source:
                    recovered_counts[class_name] += int(
                        publish_member(source, destination, member.size)
                    )

    return recovered_counts


# ------------------------------------------------------------------
# Archive integrity and remote-source inventory
# ------------------------------------------------------------------

ARCHIVE_SOURCE_COLUMNS = [
    "source_id", "source_url", "source_relative_path", "local_path", "filename"
]
ARCHIVE_REPAIR_COLUMNS = [
    "archive_path", "source_url", "initial_problem", "final_status",
    "relevant_classes", "relevant_class_count", "failure_category",
    "download_attempts", "replacement_downloaded", "recovered_images",
    "quarantine_path", "candidate_path", "error_type", "error_message",
]
IMAGE_REPLACEMENT_COLUMNS = [
    "archive_path", "image_path", "quarantine_path", "reason"
]


def archive_error_category(error):
    """Only integrity/read failures can trigger archive replacement."""
    if isinstance(error, (tarfile.CompressionError, NotImplementedError)):
        return "unsupported_compression"
    if isinstance(error, (
        zipfile.BadZipFile, tarfile.ReadError, tarfile.HeaderError,
        gzip.BadGzipFile, EOFError, zlib.error, lzma.LZMAError,
    )):
        return "archive_integrity_error"
    if isinstance(error, OSError):
        return "filesystem_error"
    if isinstance(error, RuntimeError):
        # Includes encrypted ZIP members and missing compression modules.
        return "archive_environment_error"
    return "operation_error"


def validate_archive_integrity(archive_path):
    """Reads archive contents, including ZIP CRCs and compressed TAR trailers."""
    archive_path = Path(archive_path)
    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            if any(member.flag_bits & 1 for member in archive.infolist()):
                raise RuntimeError("Encrypted ZIP members require a password.")
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f"CRC/header failure in member: {bad_member}")
    elif archive_path.name.lower().endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:*") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue
                source = archive.extractfile(member)
                if source is None:
                    raise EOFError(f"Unreadable TAR member: {member.name}")
                with source:
                    total = sum(len(chunk) for chunk in iter(
                        lambda: source.read(1024 * 1024), b""
                    ))
                if total != member.size:
                    raise EOFError(f"Truncated TAR member: {member.name}")
            # Force compressed-stream EOF/CRC checks even after TAR end markers.
            while archive.fileobj.read(1024 * 1024):
                pass
    else:
        raise ValueError(f"Unsupported archive suffix: {archive_path.name}")


def build_drive_archive_source_manifest(config, dirs):
    """Lists remote archive IDs/paths without downloading their contents."""
    import gdown

    dataset_root = Path(dirs["dataset_root_dir"]).resolve()
    entries = gdown.download_folder(
        url=config["dataset_google_drive_folder_url"],
        output=str(dataset_root),
        quiet=True,
        skip_download=True,
    )
    if entries is None:
        raise RuntimeError("gdown could not list the source Google Drive folder.")

    records = []
    for entry in entries:
        if not hasattr(entry, "id") or not hasattr(entry, "path"):
            raise RuntimeError("Unsupported gdown folder-listing result; update gdown.")
        relative_path = Path(str(entry.path).replace("\\", "/"))
        if not relative_path.name.lower().endswith((".zip", ".tar.gz", ".tgz")):
            continue
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError(f"Unsafe remote archive path: {entry.path}")
        source_id = str(entry.id)
        if not re.fullmatch(r"[A-Za-z0-9_-]+", source_id):
            raise ValueError("gdown returned an invalid Google Drive file ID.")
        local_path = (dataset_root / relative_path).resolve()
        if not local_path.is_relative_to(dataset_root):
            raise ValueError(f"Archive path escapes the dataset directory: {entry.path}")
        records.append({
            "source_id": source_id,
            "source_url": f"https://drive.google.com/uc?id={source_id}",
            "source_relative_path": relative_path.as_posix(),
            "local_path": str(local_path),
            "filename": relative_path.name,
        })

    result = pd.DataFrame.from_records(records, columns=ARCHIVE_SOURCE_COLUMNS)
    if result["source_relative_path"].duplicated().any():
        raise ValueError("Source Drive folder has duplicate archive paths; mapping is ambiguous.")
    return result.sort_values("source_relative_path", kind="stable").reset_index(drop=True)


def match_archive_source(archive_path, source_manifest):
    """Prefers exact local paths; permits a filename match only when unique."""
    archive_path = Path(archive_path).resolve()
    matches = source_manifest.loc[source_manifest["local_path"].eq(str(archive_path))]
    if matches.empty:
        matches = source_manifest.loc[
            source_manifest["filename"].str.casefold().eq(archive_path.name.casefold())
        ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one remote source for {archive_path.name}; found {len(matches)}. "
            "No replacement was downloaded."
        )
    return matches.iloc[0].to_dict()


def quarantine_dataset_file(path, raw_root, quarantine_dir):
    """Moves one exact file to quarantine outside the raw dataset tree."""
    path = Path(path)
    raw_root = Path(raw_root).resolve()
    resolved = path.resolve()
    if path.is_symlink() or not path.is_file() or not resolved.is_relative_to(raw_root):
        raise ValueError(f"Cannot quarantine a non-regular/out-of-dataset path: {path}")
    quarantine_dir = Path(quarantine_dir).resolve()
    if quarantine_dir.is_relative_to(raw_root):
        raise ValueError("Quarantine must be outside the raw-dataset scan root.")
    destination = (
        quarantine_dir / "originals" / uuid4().hex / resolved.relative_to(raw_root)
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    path.rename(destination)
    return destination


def save_provisioning_table(table, path):
    """Publishes a CSV after its temporary write completes."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.part")
    try:
        table.to_csv(temporary, index=False)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def run_labelled_archive_recovery(config, dirs, expected_class_counts):
    """
    Recovers local images, with at most ONE replacement download per
    source archive in this invocation. Counts alone never trigger
    redownload of a present, readable archive.

    Returns report tables, output locations, and operation flags.
    No full-folder redownload is performed by this function.
    """
    raw_root = Path(dirs["raw_data_dir"]).resolve()
    labelled_dir = Path(dirs["labelled_images_dir"]).resolve()
    execution_id = uuid4().hex
    reports_dir = Path(dirs["reports_dir"]) / "archive_repair" / execution_id
    quarantine_dir = raw_root.parent / f"{raw_root.name}_quarantine" / execution_id
    records = []
    replacements = []
    source_manifest = pd.DataFrame(columns=ARCHIVE_SOURCE_COLUMNS)
    source_loaded = False
    source_error = None
    processed_paths = set()
    attempted_source_ids = set()
    recovery_attempted = False
    repair_allowed = bool(
        config["dataset_download_enabled"]
        and config.get("dataset_archive_repair_enabled", True)
    )

    def persist_reports():
        save_provisioning_table(
            pd.DataFrame.from_records(records, columns=ARCHIVE_REPAIR_COLUMNS),
            reports_dir / "archive_repair_report.csv",
        )
        save_provisioning_table(
            pd.DataFrame.from_records(replacements, columns=IMAGE_REPLACEMENT_COLUMNS),
            reports_dir / "image_replacement_report.csv",
        )
        save_provisioning_table(source_manifest, reports_dir / "archive_source_manifest.csv")

    def sources():
        nonlocal source_loaded, source_manifest, source_error
        if not source_loaded:
            source_loaded = True
            try:
                print("Listing source archives in Google Drive (no file download)...")
                source_manifest = build_drive_archive_source_manifest(config, dirs)
            except Exception as error:
                source_error = error
        if source_error is not None:
            raise RuntimeError(f"Archive-source discovery failed: {source_error}") from source_error
        return source_manifest

    def remaining_classes():
        scan = scan_labelled_image_classes(labelled_dir, expected_class_counts)
        return set(scan["missing_classes"]) | set(scan["incomplete_classes"])

    def replace_archive(path, row, known_source):
        if not repair_allowed:
            raise RuntimeError("Archive repair/download is disabled in CONFIG.")
        source = known_source or match_archive_source(path, sources())
        row["source_url"] = source["source_url"]
        if source["source_id"] in attempted_source_ids:
            raise RuntimeError("The single download attempt for this source archive was already used.")
        gdown_executable = shutil.which("gdown")
        if gdown_executable is None:
            raise RuntimeError("gdown is not available on PATH; run %pip install -q gdown.")

        # Stage the replacement outside all raw-dataset/archive scans.
        candidate = quarantine_dir / "downloads" / uuid4().hex / path.name
        candidate.parent.mkdir(parents=True, exist_ok=True)
        row["candidate_path"] = str(candidate)
        row["download_attempts"] = 1
        attempted_source_ids.add(source["source_id"])
        persist_reports()
        print(f"Downloading one archive replacement: {path.name}")
        run_gdown_download([
            gdown_executable, source["source_url"], "-O", str(candidate)
        ])
        if not candidate.is_file():
            raise RuntimeError("gdown returned without creating the replacement archive.")
        validate_archive_integrity(candidate)

        # Do not move/replace the old archive until the new one is verified.
        if path.exists():
            row["quarantine_path"] = str(
                quarantine_dataset_file(path, raw_root, quarantine_dir)
            )
        path.parent.mkdir(parents=True, exist_ok=True)
        candidate.replace(path)
        row["replacement_downloaded"] = True

    def process_archive(path, known_source=None):
        nonlocal recovery_attempted
        path = Path(path).resolve()
        if not path.is_relative_to(raw_root):
            raise ValueError(f"Archive is outside the declared raw-dataset root: {path}")
        if path in processed_paths:
            return
        processed_paths.add(path)
        targets = remaining_classes()
        if not targets:
            return
        row = {
            "archive_path": str(path), "source_url": "", "initial_problem": "",
            "final_status": "in_progress", "download_attempts": 0,
            "relevant_classes": "", "relevant_class_count": 0, "failure_category": "",
            "replacement_downloaded": False, "recovered_images": 0,
            "quarantine_path": "", "candidate_path": "",
            "error_type": "", "error_message": "",
        }
        records.append(row)
        persist_reports()

        try:
            needs_replacement = not path.exists()
            relevant_classes = set()
            if needs_replacement:
                row["initial_problem"] = "missing_archive"
            else:
                try:
                    relevant_classes = inspect_archive_for_classes(path, targets)
                    row["relevant_classes"] = " | ".join(sorted(relevant_classes))
                    row["relevant_class_count"] = len(relevant_classes)
                    if not relevant_classes:
                        row["final_status"] = "not_relevant"
                        return
                    recovery_attempted = True
                    print(f"Checking archive contents: {path.name}")
                    validate_archive_integrity(path)
                except Exception as error:
                    row["initial_problem"] = archive_error_category(error)
                    if row["initial_problem"] != "archive_integrity_error":
                        raise
                    needs_replacement = True

            if needs_replacement:
                recovery_attempted = True
                replace_archive(path, row, known_source)
                relevant_classes = inspect_archive_for_classes(path, targets)
            if not relevant_classes:
                row["final_status"] = "not_relevant"
                return

            row["relevant_classes"] = " | ".join(sorted(relevant_classes))
            row["relevant_class_count"] = len(relevant_classes)
            for extraction_pass in range(2):
                try:
                    print(f"Recovering images from: {path.name}")
                    counts = recover_required_images_from_archive(
                        archive_path=path, labelled_images_dir=labelled_dir,
                        target_classes=relevant_classes, raw_root=raw_root,
                        quarantine_dir=quarantine_dir, replacement_records=replacements,
                    )
                    break
                except Exception as error:
                    category = archive_error_category(error)
                    if (
                        category != "archive_integrity_error"
                        or row["download_attempts"] != 0
                        or extraction_pass != 0
                    ):
                        raise
                    row["initial_problem"] = category
                    replace_archive(path, row, known_source)
            row["recovered_images"] = sum(counts.values())
            row["final_status"] = "recovered" if row["recovered_images"] else "reused"

        except Exception as error:
            row["final_status"] = "failed"
            row["error_type"] = type(error).__name__
            row["error_message"] = str(error)
            row["failure_category"] = archive_error_category(error)
            print(f"Archive operation failed: {path.name}: {error}")
        except BaseException:
            row["final_status"] = "interrupted"
            raise
        finally:
            persist_reports()

    try:
        for archive_path in find_dataset_archives(raw_root):
            if not remaining_classes():
                break
            process_archive(archive_path)
            # Storage/tool/format errors need correction, not more downloads.
            if records and records[-1]["final_status"] == "failed":
                if records[-1]["failure_category"] in {
                    "filesystem_error", "archive_environment_error", "unsupported_compression"
                }:
                    break

        # A missing archive has no local path to discover. Consult the remote
        # inventory for absent class archives / labelled-image bundle archives.
        # A valid archive already present is never downloaded again on counts alone.
        targets = remaining_classes()
        if targets and repair_allowed and not any(row["final_status"] == "failed" for row in records):
            try:
                remote_sources = sources()
                bundle_key = normalize_class_key(Path(dirs["labelled_images_dir"]).name)
                for source in remote_sources.to_dict(orient="records"):
                    targets = remaining_classes()
                    if not targets:
                        break
                    remote_path = Path(source["source_relative_path"])
                    parts = {normalize_class_key(part) for part in remote_path.parts}
                    class_keys = {normalize_class_key(name) for name in targets}
                    relevant = bool(parts & class_keys) or normalize_class_key(remote_path.name) == bundle_key
                    if not relevant:
                        continue
                    expected_path = Path(source["local_path"])
                    # Reuse a uniquely matching archive stored elsewhere in raw.
                    local_matches = [
                        path for path in find_dataset_archives(raw_root)
                        if path.name.casefold() == remote_path.name.casefold()
                    ]
                    if expected_path.exists() or local_matches:
                        continue
                    process_archive(expected_path, known_source=source)
            except Exception as error:
                records.append({
                    "archive_path": "", "source_url": "", "initial_problem": "source_discovery_error",
                    "final_status": "failed", "download_attempts": 0,
                    "relevant_classes": "", "relevant_class_count": 0,
                    "failure_category": "source_discovery_error",
                    "replacement_downloaded": False, "recovered_images": 0,
                    "quarantine_path": "", "candidate_path": "",
                    "error_type": type(error).__name__, "error_message": str(error),
                })
    finally:
        persist_reports()

    return {
        "archive_repair_report": pd.DataFrame.from_records(records, columns=ARCHIVE_REPAIR_COLUMNS),
        "archive_source_manifest": source_manifest,
        "image_replacement_report": pd.DataFrame.from_records(replacements, columns=IMAGE_REPLACEMENT_COLUMNS),
        "archive_recovery_attempted": recovery_attempted,
        "archive_recovery_performed": any(row["recovered_images"] > 0 for row in records),
        "archive_redownload_performed": any(row["replacement_downloaded"] for row in records),
        "operations_succeeded": all(row["final_status"] != "failed" for row in records),
        "reports_dir": reports_dir,
        "quarantine_dir": quarantine_dir,
    }


# ------------------------------------------------------------------
# Ensure presence BEFORE building the initial inventory -- NEW
# ------------------------------------------------------------------

# Clear any successful flag left by a previous notebook execution.
RAW_DATASET_AVAILABLE = False
DOWNLOAD_STATUS = ensure_raw_dataset_present(config=CONFIG, dirs=DIRS)

DATASET_PROVISIONING_METHOD = {
    "already_present": "existing_storage",
    "downloaded": "gdown_google_drive",
}[DOWNLOAD_STATUS]


# ------------------------------------------------------------------
# Initial inventory and class scan
# ------------------------------------------------------------------

RAW_DATASET_INVENTORY = build_raw_dataset_inventory(config=CONFIG, dirs=DIRS)

LABELLED_IMAGE_SCAN = scan_labelled_image_classes(
    labelled_images_dir=labelled_images_dir,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)
labelled_class_report = build_labelled_class_report(
    scan_report=LABELLED_IMAGE_SCAN,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

print("Initial labelled-image inventory:")
print(
    "Classes:",
    f"{LABELLED_IMAGE_SCAN['existing_class_count']}/{EXPECTED_LABELLED_CLASS_COUNT}",
)
print(
    "Images:",
    f"{LABELLED_IMAGE_SCAN['total_actual']:,}/{EXPECTED_LABELLED_IMAGE_COUNT:,}",
)
display(labelled_class_report)


# ------------------------------------------------------------------
# Recover labelled images, repairing damaged/missing source archives
# ------------------------------------------------------------------

ARCHIVE_RECOVERY_RESULT = run_labelled_archive_recovery(
    config=CONFIG,
    dirs=DIRS,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

archive_repair_report = ARCHIVE_RECOVERY_RESULT["archive_repair_report"]
archive_source_manifest = ARCHIVE_RECOVERY_RESULT["archive_source_manifest"]
image_replacement_report = ARCHIVE_RECOVERY_RESULT["image_replacement_report"]
ARCHIVE_RECOVERY_ATTEMPTED = ARCHIVE_RECOVERY_RESULT["archive_recovery_attempted"]
ARCHIVE_RECOVERY_PERFORMED = ARCHIVE_RECOVERY_RESULT["archive_recovery_performed"]
ARCHIVE_REDOWNLOAD_PERFORMED = ARCHIVE_RECOVERY_RESULT["archive_redownload_performed"]
PROVISIONING_REPORTS_DIR = ARCHIVE_RECOVERY_RESULT["reports_dir"]

archive_inventory_report = (
    archive_repair_report.loc[:, [
        "archive_path", "relevant_classes", "relevant_class_count"
    ]]
    .rename(columns={"archive_path": "archive"})
)
display(archive_repair_report)


# ------------------------------------------------------------------
# Final inventory: always rebuild after any download / recovery
# ------------------------------------------------------------------

LABELLED_IMAGE_SCAN = scan_labelled_image_classes(
    labelled_images_dir=labelled_images_dir,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)
labelled_class_report = build_labelled_class_report(
    scan_report=LABELLED_IMAGE_SCAN,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

print("\nFinal labelled-image inventory:")
display(labelled_class_report)

RAW_DATASET_INVENTORY = {
    **build_raw_dataset_inventory(config=CONFIG, dirs=DIRS),
    "labelled_class_count": LABELLED_IMAGE_SCAN["existing_class_count"],
    "expected_labelled_class_count": EXPECTED_LABELLED_CLASS_COUNT,
    "expected_labelled_class_count_met": (
        LABELLED_IMAGE_SCAN["existing_class_count"] == EXPECTED_LABELLED_CLASS_COUNT
    ),
    "expected_labelled_classes_present": LABELLED_IMAGE_SCAN[
        "expected_classes_present"
    ],
    "expected_class_image_counts_met": LABELLED_IMAGE_SCAN[
        "expected_class_counts_met"
    ],
    "expected_labelled_image_count_met": LABELLED_IMAGE_SCAN[
        "expected_total_count_met"
    ],
    "missing_labelled_classes": LABELLED_IMAGE_SCAN["missing_classes"],
    "incomplete_labelled_classes": LABELLED_IMAGE_SCAN["incomplete_classes"],
    "excess_labelled_classes": LABELLED_IMAGE_SCAN["excess_classes"],
    "unexpected_labelled_classes": LABELLED_IMAGE_SCAN["unexpected_classes"],
    "archive_recovery_attempted": ARCHIVE_RECOVERY_ATTEMPTED,
    "archive_recovery_performed": ARCHIVE_RECOVERY_PERFORMED,
    "archive_redownload_performed": ARCHIVE_REDOWNLOAD_PERFORMED,
    "archive_recovery_operations_succeeded": ARCHIVE_RECOVERY_RESULT["operations_succeeded"],
    "dataset_provisioning_method": DATASET_PROVISIONING_METHOD,
    "dataset_download_status": DOWNLOAD_STATUS,
    "dataset_download_performed": DOWNLOAD_STATUS == "downloaded",
}

RAW_DATASET_CHECK_COLUMNS = [
    "raw_data_dir_exists",
    "dataset_root_dir_exists",
    "labelled_images_dir_exists",
    "metadata_available",
    "minimum_video_count_met",
    "minimum_labelled_image_count_met",
    "expected_labelled_class_count_met",
    "expected_labelled_classes_present",
    "expected_class_image_counts_met",
    "expected_labelled_image_count_met",
    "archive_recovery_operations_succeeded",
]

raw_dataset_inventory_report = pd.DataFrame.from_records([RAW_DATASET_INVENTORY])

raw_dataset_validation_report = (
    raw_dataset_inventory_report[RAW_DATASET_CHECK_COLUMNS]
    .T
    .rename(columns={0: "check_passed"})
    .rename_axis("validation_check")
    .reset_index()
    .assign(check_passed=lambda data: data["check_passed"].astype("boolean"))
)
failed_dataset_checks = (
    raw_dataset_validation_report.loc[
        ~raw_dataset_validation_report["check_passed"].fillna(False)
    ]
    .reset_index(drop=True)
)
RAW_DATASET_AVAILABLE = bool(
    raw_dataset_validation_report["check_passed"].fillna(False).all()
)

# Keep the aggregate flag consistent with ALL final checks.
RAW_DATASET_INVENTORY = {
    **RAW_DATASET_INVENTORY,
    "validation_passed": RAW_DATASET_AVAILABLE,
}
raw_dataset_inventory_report = raw_dataset_inventory_report.assign(
    validation_passed=RAW_DATASET_AVAILABLE
)

display(raw_dataset_inventory_report.T.rename(columns={0: "value"}))
display(raw_dataset_validation_report)

# Save the final diagnostics BEFORE the validation gate, including failures.
save_provisioning_table(
    labelled_class_report, PROVISIONING_REPORTS_DIR / "labelled_class_report.csv"
)
save_provisioning_table(
    raw_dataset_validation_report, PROVISIONING_REPORTS_DIR / "raw_dataset_validation_report.csv"
)
save_provisioning_table(
    raw_dataset_inventory_report, PROVISIONING_REPORTS_DIR / "raw_dataset_inventory_report.csv"
)
print("Provisioning reports saved to:", PROVISIONING_REPORTS_DIR)

if not RAW_DATASET_AVAILABLE:
    failed_check_names = failed_dataset_checks["validation_check"].tolist()
    raise RuntimeError(
        "Kvasir-Capsule did not pass final content validation "
        "after optional download and archive recovery. "
        f"Provisioning: {DATASET_PROVISIONING_METHOD}. "
        f"Failed checks: {failed_check_names}. "
        f"Missing labelled classes: {LABELLED_IMAGE_SCAN['missing_classes']}. "
        "Incomplete labelled classes: "
        f"{list(LABELLED_IMAGE_SCAN['incomplete_classes'])}. "
        f"Excess labelled classes: {list(LABELLED_IMAGE_SCAN['excess_classes'])}. "
        f"Inspect the reports and dataset contents at: {dataset_root_dir}. "
        "A count mismatch is a validation finding, not proof that another "
        "full download is required. "
        f"Reports: {PROVISIONING_REPORTS_DIR}"
    )

print("Raw dataset provisioning:", DATASET_PROVISIONING_METHOD)
print("Download status:", DOWNLOAD_STATUS)
print("Archive recovery attempted:", ARCHIVE_RECOVERY_ATTEMPTED)
print("Archive recovery performed:", ARCHIVE_RECOVERY_PERFORMED)
print("Archive redownload performed:", ARCHIVE_REDOWNLOAD_PERFORMED)
print("Raw dataset available:", RAW_DATASET_AVAILABLE)
print("Discovered videos:", f"{RAW_DATASET_INVENTORY['video_count']:,}")
print(
    "Discovered labelled classes:",
    f"{LABELLED_IMAGE_SCAN['existing_class_count']}/{EXPECTED_LABELLED_CLASS_COUNT}",
)
print(
    "Discovered labelled images:",
    f"{LABELLED_IMAGE_SCAN['total_actual']:,}/{EXPECTED_LABELLED_IMAGE_COUNT:,}",
)
print("Metadata path:", DIRS["metadata_path"])


Existing dataset files found at: /content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule/osfstorage
Skipping download; continuing with content validation.
Initial labelled-image inventory:
Classes: 14/14
Images: 47,238/47,238


,class_name,actual_images,expected_images,difference,status
0,Ampulla of vater,10,10,0,complete
1,Angiectasia,866,866,0,complete
2,Blood - fresh,446,446,0,complete
3,Blood - hematin,12,12,0,complete
4,Erosion,506,506,0,complete
5,Erythema,159,159,0,complete
6,Foreign body,776,776,0,complete
7,Ileocecal valve,4189,4189,0,complete
8,Lymphangiectasia,592,592,0,complete
9,Normal clean mucosa,34338,34338,0,complete


,archive_path,source_url,initial_problem,final_status,relevant_classes,relevant_class_count,failure_category,download_attempts,replacement_downloaded,recovered_images,quarantine_path,candidate_path,error_type,error_message



Final labelled-image inventory:


,class_name,actual_images,expected_images,difference,status
0,Ampulla of vater,10,10,0,complete
1,Angiectasia,866,866,0,complete
2,Blood - fresh,446,446,0,complete
3,Blood - hematin,12,12,0,complete
4,Erosion,506,506,0,complete
5,Erythema,159,159,0,complete
6,Foreign body,776,776,0,complete
7,Ileocecal valve,4189,4189,0,complete
8,Lymphangiectasia,592,592,0,complete
9,Normal clean mucosa,34338,34338,0,complete


,value
raw_data_dir,/content/drive/MyDrive/MMVQA_Clinical/data/raw...
dataset_root_dir,/content/drive/MyDrive/MMVQA_Clinical/data/raw...
labelled_images_dir,/content/drive/MyDrive/MMVQA_Clinical/data/raw...
metadata_path,/content/drive/MyDrive/MMVQA_Clinical/data/raw...
raw_data_dir_exists,True
dataset_root_dir_exists,True
labelled_images_dir_exists,True
metadata_available,True
video_count,117
minimum_video_files,117


,validation_check,check_passed
0,raw_data_dir_exists,True
1,dataset_root_dir_exists,True
2,labelled_images_dir_exists,True
3,metadata_available,True
4,minimum_video_count_met,True
5,minimum_labelled_image_count_met,True
6,expected_labelled_class_count_met,True
7,expected_labelled_classes_present,True
8,expected_class_image_counts_met,True
9,expected_labelled_image_count_met,True


Provisioning reports saved to: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/reports/archive_repair/77cdbdc368474896aeeb3788affc8d30
Raw dataset provisioning: existing_storage
Download status: already_present
Archive recovery attempted: False
Archive recovery performed: False
Archive redownload performed: False
Raw dataset available: True
Discovered videos: 117
Discovered labelled classes: 14/14
Discovered labelled images: 47,238/47,238
Metadata path: /content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule/osfstorage/metadata.csv


### 7. Normalization layer

In [ ]:
COLUMN_NAME_RULES = (
    (re.compile(r"[^a-z0-9]+"), "_"),
    (re.compile(r"_+"), "_"),
)

LABEL_RULES = (
    (re.compile(r"[_\-/]+"), " "),
    (re.compile(r"\s+"), " "),
)

COLUMN_ALIASES = {
    "file_name": "filename",
    "image_name": "filename",
    "image_filename": "filename",
    "video": "video_id",
    "video_name": "video_id",
    "frame": "frame_number",
    "frame_no": "frame_number",
    "label": "finding_class",
    "class": "finding_class",
    "category": "finding_category",
}

BBOX_SPEC = {
    "x_columns": ("x1", "x2", "x3", "x4"),
    "y_columns": ("y1", "y2", "y3", "y4"),
}

REQUIRED_COLUMNS = (
    "filename",
    "video_id",
    "finding_class",
)


def is_missing_scalar(value):
    """Returns True only for scalar missing values."""

    if value is None:
        return True

    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False

    return isinstance(
        missing,
        (bool, np.bool_),
    ) and bool(missing)


def scalar_to_text(value):
    """
    Converts a scalar metadata value to clean text.

    Sequences are rejected because Phase 2 metadata fields are
    expected to be scalar. This prevents silent label loss.
    """

    if isinstance(value, (list, tuple)):
        raise TypeError(
            "Expected a scalar metadata value, "
            f"but received {type(value).__name__}."
        )

    if is_missing_scalar(value):
        return ""

    return str(value).strip()


def apply_text_rules(
    text,
    rules,
):
    """Applies ordered regex transformations."""

    result = text

    for pattern, replacement in rules:
        result = pattern.sub(
            replacement,
            result,
        )

    return result


def apply_series_rules(
    series,
    rules,
):
    """Vectorized equivalent for a pandas Series."""

    result = series

    for pattern, replacement in rules:
        result = result.str.replace(
            pattern,
            replacement,
            regex=True,
        )

    return result


def normalize_column_name(value):
    """Creates a canonical snake_case column name."""

    text = scalar_to_text(value).casefold()

    return apply_text_rules(
        text=text,
        rules=COLUMN_NAME_RULES,
    ).strip("_")

def normalize_id(value):
    """Creates a filename-independent matching key."""

    text = Path(
        scalar_to_text(value)
    ).stem

    return re.sub(
        pattern=r"[^a-zA-Z0-9]+",
        repl="",
        string=text,
    ).casefold()

In [ ]:
def canonicalize_columns(
    dataframe,
    aliases,
):
    """
    Returns a new DataFrame with normalized and canonical columns.

    Raises an error if multiple source columns resolve to the
    same canonical column.
    """

    normalized_columns = [
        normalize_column_name(column)
        for column in dataframe.columns
    ]

    canonical_columns = [
        aliases.get(column, column)
        for column in normalized_columns
    ]

    canonical_index = pd.Index(
        canonical_columns
    )

    duplicate_columns = (
        canonical_index[
            canonical_index.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_columns:
        raise ValueError(
            "Multiple metadata columns resolve to the same "
            f"canonical name: {duplicate_columns}"
        )

    result = dataframe.copy()
    result.columns = canonical_columns

    return result


def validate_required_columns(
    dataframe,
    required_columns,
):
    """Validates the input schema without modifying it."""

    missing_columns = sorted(
        set(required_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Missing required metadata columns: "
            f"{missing_columns}"
        )

    return dataframe

In [ ]:
def add_label_columns(
    dataframe,
    clinical_group_map,
    strict=True,
):
    """
    Adds clean, normalized, and grouped label columns.

    The source label is preserved in finding_class_raw.
    """

    source_labels = (
        dataframe["finding_class_raw"]
        if "finding_class_raw" in dataframe.columns
        else dataframe["finding_class"]
    )

    clean_labels = source_labels.map(
        scalar_to_text
    )

    normalized_labels = apply_series_rules(
        series=(
            clean_labels
            .str.casefold()
            .str.strip()
        ),
        rules=LABEL_RULES,
    ).str.strip()

    clinical_groups = normalized_labels.map(
        clinical_group_map
    )

    unmapped_labels = sorted(
        normalized_labels[
            normalized_labels.ne("")
            & clinical_groups.isna()
        ]
        .unique()
        .tolist()
    )

    if strict and unmapped_labels:
        raise ValueError(
            "Labels missing from clinical_group_map: "
            f"{unmapped_labels}"
        )

    return dataframe.assign(
        finding_class_raw=source_labels,
        finding_class=clean_labels,
        finding_class_normalized=normalized_labels,
        clinical_group=clinical_groups,
    )


def add_matching_keys(dataframe):
    """Adds normalized image and video matching keys."""

    return dataframe.assign(
        image_key=(
            dataframe["filename"]
            .map(normalize_id)
        ),
        video_key=(
            dataframe["video_id"]
            .map(normalize_id)
        ),
    )

In [ ]:
def add_empty_bbox_columns(dataframe):
    """Returns the stable bbox schema for data without annotations."""

    index = dataframe.index

    return dataframe.assign(
        bbox_annotation_present=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_complete=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_invalid=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        has_bbox=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_xmin=np.nan,
        bbox_ymin=np.nan,
        bbox_xmax=np.nan,
        bbox_ymax=np.nan,
        bbox_width=np.nan,
        bbox_height=np.nan,
        bbox_area=np.nan,
    )


def normalize_bounding_boxes(
    dataframe,
    bbox_spec,
):
    """
    Adds a validated axis-aligned bounding-box representation.

    Original coordinate columns are not overwritten.
    """

    x_columns = tuple(
        bbox_spec["x_columns"]
    )

    y_columns = tuple(
        bbox_spec["y_columns"]
    )

    expected_columns = (
        x_columns
        + y_columns
    )

    present_columns = tuple(
        column
        for column in expected_columns
        if column in dataframe.columns
    )

    if not present_columns:
        return add_empty_bbox_columns(
            dataframe
        )

    missing_schema_columns = sorted(
        set(expected_columns)
        - set(present_columns)
    )

    if missing_schema_columns:
        raise ValueError(
            "Incomplete bounding-box schema. "
            f"Missing columns: {missing_schema_columns}"
        )

    coordinates = (
        dataframe
        .loc[:, expected_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    annotation_present = (
        coordinates
        .notna()
        .any(axis=1)
    )

    bbox_complete = (
        coordinates
        .notna()
        .all(axis=1)
    )

    xmin = coordinates[
        list(x_columns)
    ].min(axis=1)

    xmax = coordinates[
        list(x_columns)
    ].max(axis=1)

    ymin = coordinates[
        list(y_columns)
    ].min(axis=1)

    ymax = coordinates[
        list(y_columns)
    ].max(axis=1)

    geometry_valid = (
        bbox_complete
        & xmax.gt(xmin)
        & ymax.gt(ymin)
    )

    width = (
        xmax - xmin
    ).where(geometry_valid)

    height = (
        ymax - ymin
    ).where(geometry_valid)

    return dataframe.assign(
        bbox_annotation_present=annotation_present,
        bbox_complete=bbox_complete,
        bbox_invalid=(
            annotation_present
            & ~geometry_valid
        ),
        has_bbox=geometry_valid,
        bbox_xmin=xmin.where(geometry_valid),
        bbox_ymin=ymin.where(geometry_valid),
        bbox_xmax=xmax.where(geometry_valid),
        bbox_ymax=ymax.where(geometry_valid),
        bbox_width=width,
        bbox_height=height,
        bbox_area=width * height,
    )

In [ ]:
def normalize_metadata(
    dataframe,
    clinical_group_map,
):
    """
    Pure-by-contract Phase 2 metadata normalization pipeline.

    Input:
        raw DataFrame

    Output:
        new normalized DataFrame
    """

    return (
        dataframe
        .pipe(
            canonicalize_columns,
            aliases=COLUMN_ALIASES,
        )
        .pipe(
            validate_required_columns,
            required_columns=REQUIRED_COLUMNS,
        )
        .pipe(
            add_label_columns,
            clinical_group_map=clinical_group_map,
            strict=True,
        )
        .pipe(
            add_matching_keys,
        )
        .pipe(
            normalize_bounding_boxes,
            bbox_spec=BBOX_SPEC,
        )
    )


df = normalize_metadata(
    dataframe=df_raw,
    clinical_group_map=(
        CONFIG["clinical_group_map"]
    ),
)

### 8. Dataset Validation, metadata Cleaning, and audit

In [ ]:
# Column names describe the input schema, not expected dataset contents.
FRAME_COLUMNS = ["video_key", "frame_number"]
CLASS_COLUMN = "finding_class_normalized"
MATCH_COLUMNS = ["_audit_image", "_audit_class"]
DATASET_CHARACTERISTIC_REPORT_COLUMNS = [
    "characteristic", "actual_value", "expected_value", "validation_passed",
]
DISPLAY_MAX_ROWS = 30

# Storage is declarative: only clean metadata uses Parquet. CSV is used for
# flat reports; JSON preserves the schema and values of annotation diagnostics.
# JSON-valued CSV cells store lists of classes as readable JSON arrays.
AUDIT_STORAGE_SPEC = pd.DataFrame.from_records(
    [
        ("metadata_clean", "parquet", (), ("dataframe_clean",),
         ["image_key", *FRAME_COLUMNS, CLASS_COLUMN]),
        ("physical_image_inventory", "csv", ("matching_classes",), ("physical_image_inventory",),
         ["relative_path", "filename", "_audit_filename", "_audit_image", "_audit_class"]),
        ("dataset_characteristics", "csv", (), ("characteristics",),
         ["characteristic", "actual_value"]),
        ("dataset_validation", "csv", (), ("validation",), DATASET_CHARACTERISTIC_REPORT_COLUMNS),
        ("duplicate_annotations", "json", (), ("diagnostics", "duplicate_annotations"),
         ["image_key", "_audit_source_row", "_audit_removed", "_audit_duplicate_group"]),
        ("multi_class_frames", "csv", ("classes",), ("diagnostics", "multi_class_frames"),
         [*FRAME_COLUMNS, "class_count", "classes", "annotation_count"]),
        ("multi_annotation_same_class_groups", "csv", (), ("diagnostics", "multi_annotation_same_class_groups"),
         [*FRAME_COLUMNS, CLASS_COLUMN, "annotation_count"]),
        ("image_key_definition", "json", (), ("diagnostics", "image_key_definition"),
         ["function", "source_column", "image_key_expression"]),
    ],
    columns=["artifact", "format", "json_columns", "audit_path", "required_columns"],
)
AUDIT_CATALOG_FILENAME = "audit_export_report.json"

# Rules compare measurements or require diagnostic tables to be empty.
# No dataset sizes, class names, video counts, or anomaly counts are assumed.
DATASET_VALIDATION_RULES = pd.DataFrame.from_records(
    [
        ("metadata_keys_reproduced_by_existing_code", "empty", "metadata_key_mismatches", None),
        ("image_keys_independent_of_row_order_and_batch", "empty", "context_dependent_image_keys", None),
        ("physical_filename_key_bijection", "empty", "physical_key_collisions", None),
        ("physical_image_keys_are_nonempty", "empty", "invalid_physical_keys", None),
        ("metadata_image_video_frame_bijection", "empty", "ambiguous_identities", None),
        ("unambiguous_class_names", "empty", "ambiguous_classes", None),
        ("nonempty_matching_keys", "empty", "invalid_matching_keys", None),
        ("physical_images_have_one_class", "empty", "unresolved_physical_classes", None),
        ("metadata_frames_exist_on_disk", "empty", "metadata_without_physical_frame", None),
        ("physical_frames_exist_in_metadata", "empty", "physical_without_metadata_frame", None),
        ("metadata_frame_class_pairs_exist_on_disk", "empty", "missing_files", None),
        ("physical_frame_class_pairs_exist_in_metadata", "empty", "missing_metadata", None),
        ("one_physical_instance_per_frame_class", "empty", "repeated_physical_files", None),
        ("no_exact_duplicates_after_cleaning", "empty", "remaining_duplicates", None),
        ("all_physical_files_accounted_for", "equal", "physical_files_total", "reconstructed_physical_files"),
        ("all_annotation_rows_accounted_for", "equal", "metadata_annotation_rows_raw", "reconstructed_raw_annotations"),
        ("all_duplicate_rows_accounted_for", "equal", "duplicate_annotation_rows_in_groups", "reconstructed_duplicate_rows"),
        ("physical_and_metadata_unique_frames_agree", "equal", "unique_labelled_frames", "unique_metadata_frames"),
        ("image_keys_and_video_frames_agree", "equal", "unique_metadata_frames", "unique_labelled_video_frames"),
        ("physical_instances_explained_by_classes", "equal", "physical_labelled_image_instances", "reconstructed_physical_instances"),
        ("clean_annotations_explained_by_multiplicity", "equal", "metadata_annotation_rows_clean", "reconstructed_clean_annotations"),
    ],
    columns=["characteristic", "operator", "left", "right"],
)


def filename_inventory_key(value):
    """Count physical filename identities independently of the generated keys."""
    name = str(value).strip().replace("\\", "/").rsplit("/", 1)[-1]
    media_type = mimetypes.guess_type(name)[0] or ""
    return (Path(name).stem if media_type.startswith("image/") else name).casefold()


def discover_image_key_builder(matching_key_function, metadata):
    """Reuse the existing image_key expression; never infer its string format.

    Supports a single .assign(image_key=lambda frame: ...) expression, an
    .assign(image_key=frame[...]) expression, or frame['image_key'] = ... .
    The expression must read one source column through frame['column'].
    Multiple source fields, multiple definitions and unavailable source code
    are reported explicitly. No guessed normalizer is used as a fallback.
    """
    function = inspect.unwrap(matching_key_function)
    try:
        source = textwrap.dedent(inspect.getsource(function))
    except (OSError, TypeError) as error:
        raise ValueError(
            "Cannot inspect the existing matching-key function. Rerun its "
            "definition cell in this notebook before running the validator."
        ) from error

    tree = ast.parse(source)
    definitions = [node for node in tree.body if isinstance(node, ast.FunctionDef)]
    if len(definitions) != 1:
        raise ValueError("Expected one inspectable matching-key function definition.")
    definition = definitions[0]
    expressions = [
        node.value for node in ast.walk(definition)
        if isinstance(node, ast.keyword) and node.arg == "image_key"
    ] + [
        node.value for node in ast.walk(definition)
        if isinstance(node, ast.Assign) and any(
            isinstance(target, ast.Subscript)
            and isinstance(target.slice, ast.Constant)
            and target.slice.value == "image_key"
            for target in node.targets
        )
    ]
    if len(expressions) != 1:
        raise ValueError(
            "Cannot select one image_key expression from the existing code. "
            "The validator will not guess between multiple definitions. "
            f"Found {len(expressions)} definitions in {function.__name__}."
        )
    expression = expressions[0]
    scope = dict(function.__globals__)
    closure = inspect.getclosurevars(function)
    scope.update(closure.nonlocals)

    if isinstance(expression, ast.Lambda):
        if len(expression.args.args) != 1 or expression.args.defaults:
            raise ValueError("The image_key lambda must accept one DataFrame argument.")
        frame_name = expression.args.args[0].arg
        body = expression.body
    else:
        frame_names = {
            node.value.id for node in ast.walk(expression)
            if isinstance(node, ast.Subscript) and isinstance(node.value, ast.Name)
            and isinstance(node.slice, ast.Constant)
            and node.slice.value in metadata.columns
        }
        if len(frame_names) != 1:
            raise ValueError("Cannot identify one DataFrame input in the image_key expression.")
        frame_name = next(iter(frame_names))
        body = expression
        expression = ast.Lambda(
            args=ast.arguments(posonlyargs=[], args=[ast.arg(arg=frame_name)],
                               kwonlyargs=[], kw_defaults=[], defaults=[]),
            body=body,
        )

    column_nodes = [
        node for node in ast.walk(body)
        if isinstance(node, ast.Subscript) and isinstance(node.value, ast.Name)
        and node.value.id == frame_name
    ]
    literal_columns = {
        node.slice.value for node in column_nodes
        if isinstance(node.slice, ast.Constant) and isinstance(node.slice.value, str)
    }
    if len(literal_columns) != 1 or any(
        not isinstance(node.slice, ast.Constant) or not isinstance(node.slice.value, str)
        for node in column_nodes
    ):
        raise ValueError(
            "The image_key expression must expose one filename source column. "
            f"Discovered columns: {sorted(literal_columns)}. Composite keys require "
            "an explicit file-to-fields mapping; they cannot be guessed from filenames."
        )
    source_column = next(iter(literal_columns))
    if source_column not in metadata.columns or source_column == "image_key":
        raise ValueError(f"Unavailable or circular source column: {source_column}")

    # All accesses to the input frame must be through the discovered column.
    # In particular, do not create physical keys from metadata row indices.
    allowed_names = {id(node.value) for node in column_nodes}
    if any(isinstance(node, ast.Name) and node.id == frame_name
           and id(node) not in allowed_names for node in ast.walk(body)):
        raise ValueError("The image_key expression accesses row/context data beyond its source column.")
    transform = eval(
        compile(ast.fix_missing_locations(ast.Expression(expression)),
                "<existing-image-key-expression>", "eval"),
        scope,
    )

    def build_keys(values):
        inputs = pd.DataFrame({source_column: pd.Series(values).reset_index(drop=True)})
        result = transform(inputs)
        if not isinstance(result, pd.Series) or not result.index.equals(inputs.index):
            raise ValueError("The discovered key expression must return one aligned Series.")
        return result.astype("string")

    source_values = metadata[source_column].reset_index(drop=True)
    generated = build_keys(source_values)
    expected = metadata["image_key"].astype("string").reset_index(drop=True)
    check = metadata[[source_column, "image_key"]].reset_index(drop=True).assign(
        _audit_generated_key=generated,
    )
    mismatches = check.loc[~generated.eq(expected).fillna(False)]

    # A filename key must retain its meaning when files are scanned in a
    # different order or annotations contain repeated rows for one filename.
    reversed_keys = build_keys(source_values.iloc[::-1]).iloc[::-1].reset_index(drop=True)
    partitions = [source_values.iloc[::2], source_values.iloc[1::2]]
    partition_keys = pd.concat([
        build_keys(partition).set_axis(partition.index)
        for partition in partitions if not partition.empty
    ]).sort_index()
    context_dependent = check.loc[
        ~generated.eq(reversed_keys).fillna(False)
        | ~generated.eq(partition_keys).fillna(False)
    ]
    discovery = pd.DataFrame.from_records([{
        "function": function.__name__,
        "source_column": source_column,
        "image_key_expression": ast.unparse(body),
    }])
    return build_keys, discovery, mismatches, context_dependent


def class_match_key(value):
    """Use the same spelling normalization for metadata labels and folder names."""
    return re.sub(r"[^a-z0-9]+", "_", str(value).casefold()).strip("_")


def scan_dataset_files(dataset_root):
    """Inventory every regular file recursively; propagate filesystem errors.

    Counts file paths, not unique bytes. Does not decode or compare image pixels.
    The selected tree should contain the labelled images, not extracted unlabelled frames.
    """
    root = Path(dataset_root).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {root}")

    records, directories = [], []

    def raise_scan_error(error):
        raise error

    # Filesystem traversal is the I/O boundary. All counts are table aggregations.
    for directory, subdirectories, filenames in os.walk(
        root, onerror=raise_scan_error, followlinks=False,
    ):
        directory = Path(directory)
        directories.append(directory.relative_to(root).as_posix())
        if any((directory / name).is_symlink() for name in subdirectories):
            raise ValueError(f"Cannot completely inventory a symlink directory: {directory}")
        for name in filenames:
            path = directory / name
            if not path.is_file():
                raise OSError(f"Cannot inventory a regular file: {path}")
            records.append({
                "relative_path": path.relative_to(root).as_posix(),
                "folder": directory.relative_to(root).as_posix(),
                "filename": name,
                "is_image": (mimetypes.guess_type(name)[0] or "").startswith("image/"),
            })

    files = pd.DataFrame.from_records(
        records, columns=["relative_path", "folder", "filename", "is_image"],
    ).astype({"is_image": "bool"})
    folder_counts = files.groupby("folder", as_index=False).agg(
        physical_files=("relative_path", "size"), physical_images=("is_image", "sum"),
    )
    folders = (
        pd.DataFrame({"folder": directories})
        .merge(folder_counts, on="folder", how="left", validate="one_to_one")
        .fillna({"physical_files": 0, "physical_images": 0})
        .astype({"physical_files": "int64", "physical_images": "int64"})
        .assign(other_files=lambda data: data["physical_files"] - data["physical_images"])
        .sort_values("folder")
        .reset_index(drop=True)
    )
    return files, folders


def attach_physical_classes(physical, metadata, dataset_root):
    """Discover class directories from observed labels and all path ancestors.

    Never discard unmatched images: they remain in the inventory and fail validation.
    Multiple matching classes also fail, rather than choosing one silently.
    """
    label_keys = metadata[["_audit_class"]].drop_duplicates()
    candidates = (
        physical[["relative_path"]]
        .assign(_audit_class=lambda data: data["relative_path"].map(
            lambda relative: [class_match_key(part) for part in
                              [Path(dataset_root).name, *Path(relative).parts[:-1]]]
        ))
        .explode("_audit_class")
        .merge(label_keys, on="_audit_class", how="inner", validate="many_to_one")
        .drop_duplicates(["relative_path", "_audit_class"])
    )
    class_matches = candidates.groupby("relative_path", as_index=False).agg(
        _audit_class=("_audit_class", "first"),
        class_match_count=("_audit_class", "size"),
        matching_classes=("_audit_class", tuple),
    )
    return (
        physical.merge(class_matches, on="relative_path", how="left", validate="one_to_one")
        .assign(class_match_count=lambda data: data["class_match_count"].fillna(0).astype("int64"))
    )


def evaluate_dataset_rules(metrics, diagnostics, rules):
    """Evaluate a declarative rule table against measured values and diagnostics."""
    unknown = set(rules["operator"]) - {"equal", "empty"}
    if unknown:
        raise ValueError(f"Unsupported validation operators: {unknown}")
    equal = rules.loc[rules["operator"].eq("equal")]
    empty = rules.loc[rules["operator"].eq("empty")]
    missing_metrics = (set(equal["left"]) | set(equal["right"])) - set(metrics.index)
    missing_tables = set(empty["left"]) - set(diagnostics)
    if missing_metrics or missing_tables:
        raise KeyError(f"Unknown metrics: {missing_metrics}; unknown diagnostics: {missing_tables}")

    comparisons = equal.assign(
        actual_value=lambda data: data["left"].map(metrics),
        expected_value=lambda data: data["right"].map(metrics),
    )
    emptiness = empty.assign(
        actual_value=lambda data: data["left"].map(lambda name: len(diagnostics[name])),
        # Zero expresses an empty diagnostic table, not a known dataset count.
        expected_value=0,
    )
    return (
        pd.concat([comparisons, emptiness]).sort_index()
        .assign(validation_passed=lambda data: data["actual_value"].eq(data["expected_value"]))
        .loc[:, DATASET_CHARACTERISTIC_REPORT_COLUMNS]
        .reset_index(drop=True)
    )


def audit_dataset(dataframe, dataset_root, matching_key_function, rules=DATASET_VALIDATION_RULES):
    """Discover dataset characteristics, clean exact duplicates, then evaluate rules.

    The input is preserved. Duplicate equality includes ALL original columns,
    including bounding boxes, and excludes the DataFrame index. Use normalized
    metadata before any previous deduplication, otherwise removed rows are unavailable.
    """
    required = {"image_key", "video_key", "frame_number", CLASS_COLUMN}
    missing = sorted(required - set(dataframe.columns))
    if missing:
        raise KeyError(f"Metadata is missing columns: {missing}")
    if dataframe.empty or not dataframe.columns.is_unique:
        raise ValueError("Metadata must be nonempty and have unique column names.")
    if any(str(column).startswith("_audit_") for column in dataframe.columns):
        raise ValueError("The '_audit_' column prefix is reserved for validation.")

    identities = dataframe[["image_key", "video_key", CLASS_COLUMN]].astype("string")
    invalid_identity = identities.apply(
        lambda column: column.isna() | column.str.strip().eq("")
    ).any(axis=1)
    numbers = pd.to_numeric(dataframe["frame_number"], errors="coerce")
    invalid_number = (
        numbers.isna() | ~np.isfinite(numbers) | numbers.lt(0) | numbers.mod(1).ne(0)
        | numbers.ge(2**63)
        | dataframe["frame_number"].map(lambda value: isinstance(value, (bool, np.bool_)))
    )
    if invalid_identity.any() or invalid_number.any():
        display(dataframe.loc[invalid_identity | invalid_number].head(DISPLAY_MAX_ROWS))
        raise ValueError(
            f"Invalid metadata: {int(invalid_identity.sum()):,} identity rows; "
            f"{int(invalid_number.sum()):,} frame-number rows."
        )

    # Compare full rows before adding audit fields. A different bbox is retained.
    duplicated = dataframe.duplicated(keep=False)
    remove = dataframe.duplicated(keep="first")
    duplicate_rows = dataframe.loc[duplicated].copy()
    duplicate_group_count = len(duplicate_rows.drop_duplicates())
    duplicate_ids = duplicate_rows.groupby(
        list(dataframe.columns), sort=False, dropna=False, observed=True,
    ).ngroup().to_numpy() + 1 if not duplicate_rows.empty else []
    duplicate_rows.insert(0, "_audit_duplicate_group", duplicate_ids)
    duplicate_rows.insert(0, "_audit_source_row", np.flatnonzero(duplicated) + 1)
    duplicate_rows.insert(1, "_audit_removed", remove.loc[duplicated].to_numpy())
    duplicate_rows = duplicate_rows.sort_values(["_audit_duplicate_group", "_audit_source_row"])
    clean = dataframe.loc[~remove].copy().reset_index(drop=True)
    build_image_keys, key_definition, key_mismatches, context_dependent_keys = (
        discover_image_key_builder(matching_key_function, clean)
    )
    work = clean.assign(
        frame_number=lambda data: pd.to_numeric(data["frame_number"]).astype("int64"),
        _audit_image=lambda data: data["image_key"].astype("string"),
        _audit_class=lambda data: data[CLASS_COLUMN].map(class_match_key),
    )

    files, folders = scan_dataset_files(dataset_root)
    physical = (
        files.loc[files["is_image"]].copy().reset_index(drop=True)
        .assign(
            _audit_filename=lambda data: data["filename"].map(filename_inventory_key),
            _audit_image=lambda data: build_image_keys(data["filename"]),
        )
        .pipe(attach_physical_classes, metadata=work, dataset_root=dataset_root)
    )
    frames = work.groupby(FRAME_COLUMNS, as_index=False, observed=True).agg(
        image_key=("image_key", "first"),
        class_count=(CLASS_COLUMN, "nunique"),
        classes=(CLASS_COLUMN, lambda values: tuple(sorted(set(values)))),
        annotation_count=(CLASS_COLUMN, "size"),
    )
    frame_classes = work.groupby(
        FRAME_COLUMNS + [CLASS_COLUMN], as_index=False, observed=True,
    ).agg(image_key=("image_key", "first"), annotation_count=("image_key", "size"))
    multi_class = frames.loc[frames["class_count"].gt(1)].copy()
    same_class = frame_classes.loc[frame_classes["annotation_count"].gt(1)].copy()

    def annotation_details(groups, keys):
        return work.merge(
            groups[keys], on=keys, how="inner", validate="many_to_one",
        ).loc[:, dataframe.columns].sort_values(FRAME_COLUMNS + [CLASS_COLUMN])

    physical_pairs = physical.groupby(
        MATCH_COLUMNS, as_index=False, observed=True, dropna=False,
    ).agg(physical_instances=("relative_path", "size"), physical_paths=("relative_path", list))
    metadata_pairs = work.groupby(MATCH_COLUMNS, as_index=False, observed=True).agg(
        metadata_annotations=("image_key", "size"),
    )
    pair_comparison = metadata_pairs.merge(
        physical_pairs, on=MATCH_COLUMNS, how="outer", indicator=True, validate="one_to_one",
    )
    physical_keys = set(physical["_audit_image"].dropna())
    metadata_keys = set(work["_audit_image"])
    identity_map = work[["image_key", "_audit_image"] + FRAME_COLUMNS].drop_duplicates()
    image_frame_counts = identity_map.drop_duplicates(
        ["_audit_image"] + FRAME_COLUMNS,
    ).groupby("_audit_image", observed=True).size()
    image_name_counts = identity_map.groupby("_audit_image", observed=True)["image_key"].nunique()
    bad_image_keys = image_frame_counts.index[image_frame_counts.gt(1)].union(
        image_name_counts.index[image_name_counts.gt(1)]
    )
    bad_frame_mask = identity_map.groupby(
        FRAME_COLUMNS, observed=True,
    )["image_key"].transform("nunique").gt(1)
    class_aliases = work[[CLASS_COLUMN, "_audit_class"]].drop_duplicates()
    physical_identities = physical[["filename", "_audit_filename", "_audit_image"]].drop_duplicates()
    physical_key_collisions = physical_identities.loc[
        physical_identities.groupby("_audit_image", dropna=False)["_audit_filename"]
        .transform("nunique").gt(1)
    ]

    diagnostics = {
        "image_key_definition": key_definition,
        "physical_image_keys": physical[["filename", "_audit_image"]].drop_duplicates(),
        "metadata_key_mismatches": key_mismatches,
        "context_dependent_image_keys": context_dependent_keys,
        "physical_key_collisions": physical_key_collisions,
        "invalid_physical_keys": physical.loc[
            physical["_audit_image"].isna() | physical["_audit_image"].str.strip().eq("")
        ],
        "physical_files_by_folder": folders,
        "duplicate_annotations": duplicate_rows,
        "multi_class_frames": multi_class,
        "multi_class_annotations": annotation_details(multi_class, FRAME_COLUMNS),
        "multi_annotation_same_class_groups": same_class,
        "multi_annotation_same_class_details": annotation_details(same_class, FRAME_COLUMNS + [CLASS_COLUMN]),
        "ambiguous_identities": identity_map.loc[identity_map["_audit_image"].isin(bad_image_keys) | bad_frame_mask],
        "ambiguous_classes": class_aliases.loc[class_aliases.duplicated("_audit_class", keep=False)],
        "invalid_matching_keys": work.loc[work[MATCH_COLUMNS].eq("").any(axis=1)],
        "unresolved_physical_classes": physical.loc[physical["class_match_count"].ne(1)],
        "metadata_without_physical_frame": work.loc[~work["_audit_image"].isin(physical_keys)].drop_duplicates("_audit_image"),
        "physical_without_metadata_frame": physical.loc[~physical["_audit_image"].isin(metadata_keys)].drop_duplicates("_audit_filename"),
        "missing_files": pair_comparison.loc[pair_comparison["_merge"].eq("left_only")],
        "missing_metadata": pair_comparison.loc[pair_comparison["_merge"].eq("right_only")],
        "repeated_physical_files": physical_pairs.loc[physical_pairs["physical_instances"].gt(1)],
        "remaining_duplicates": clean.loc[clean.duplicated(keep=False)],
    }

    measurements = pd.Series({
        "unique_labelled_frames": physical["_audit_filename"].nunique(),
        "physical_labelled_image_instances": len(physical),
        "metadata_annotation_rows_clean": len(clean),
        "multi_class_frames": len(multi_class),
        "multi_annotation_same_class_frames": len(same_class[FRAME_COLUMNS].drop_duplicates()),
        "unique_labelled_video_frames": len(frames),
        "unique_metadata_frames": work["image_key"].nunique(),
        "physical_files_total": len(files),
        "physical_non_image_files": int((~files["is_image"]).sum()),
        "metadata_annotation_rows_raw": len(dataframe),
        "duplicate_annotation_groups": duplicate_group_count,
        "duplicate_annotation_rows_in_groups": int(duplicated.sum()),
        "duplicate_annotation_rows_removed": int(remove.sum()),
        "multi_annotation_same_class_pairs": len(same_class),
        "exactly_two_annotations_same_class_frames": len(
            same_class.loc[same_class["annotation_count"].eq(2), FRAME_COLUMNS].drop_duplicates()
        ),
        "unique_metadata_frame_class_pairs": len(frame_classes),
        "extra_class_memberships": int((frames["class_count"] - 1).sum()),
        "extra_annotations_same_class": int((frame_classes["annotation_count"] - 1).sum()),
        "finding_classes": work[CLASS_COLUMN].nunique(),
        "labelled_videos": work["video_key"].nunique(),
    }, dtype="int64", name="actual_value")
    reconstructed = pd.Series({
        "reconstructed_physical_files": measurements["physical_labelled_image_instances"] + measurements["physical_non_image_files"],
        "reconstructed_raw_annotations": measurements["metadata_annotation_rows_clean"] + measurements["duplicate_annotation_rows_removed"],
        "reconstructed_duplicate_rows": measurements["duplicate_annotation_groups"] + measurements["duplicate_annotation_rows_removed"],
        "reconstructed_physical_instances": measurements["unique_labelled_video_frames"] + measurements["extra_class_memberships"],
        "reconstructed_clean_annotations": measurements["physical_labelled_image_instances"] + measurements["extra_annotations_same_class"],
    }, dtype="int64")
    return {
        "characteristics": measurements.rename_axis("characteristic").reset_index(),
        "validation": evaluate_dataset_rules(pd.concat([measurements, reconstructed]), diagnostics, rules),
        "dataframe_clean": clean,
        "diagnostics": diagnostics,
        "physical_file_inventory": files,
        "physical_image_inventory": physical,
    }


def validate_dataset_characteristics(validation_report):
    """Stop the pipeline if any declared rule fails."""
    missing = sorted(set(DATASET_CHARACTERISTIC_REPORT_COLUMNS) - set(validation_report.columns))
    if missing or validation_report.empty:
        raise ValueError(f"Empty or invalid validation report; missing columns: {missing}")
    failed = validation_report.loc[
        ~validation_report["actual_value"].eq(validation_report["expected_value"]).fillna(False)
    ]
    if not failed.empty:
        raise ValueError(
            "Dataset consistency validation failed. Inspect dataset_diagnostics. "
            f"Failed checks: {failed.to_dict(orient='records')}"
        )
    return validation_report


def audit_to_saved_tables(audit):
    """Select the persisted tables using the declarative artifact specification."""
    tables = {}
    for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
        value = audit
        for key in spec.audit_path:
            value = value[key]
        tables[spec.artifact] = value
    return tables


def saved_tables_to_audit(tables):
    """Restore the common audit structure from saved tables only."""
    audit = {"diagnostics": {}}
    for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
        target = audit
        for key in spec.audit_path[:-1]:
            target = target.setdefault(key, {})
        target[spec.audit_path[-1]] = tables[spec.artifact]
    return audit


def validate_saved_audit_tables(tables):
    """Check saved schemas, saved validation and their agreement with loaded tables.

    This checks the saved snapshot; it does not rescan the current raw dataset.
    """
    for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
        table = tables[spec.artifact]
        missing = set(spec.required_columns) - set(table.columns)
        if not table.columns.is_unique or missing:
            raise ValueError(f"Invalid schema for {spec.artifact}; missing columns: {sorted(missing)}")

    validation = tables["dataset_validation"]
    validate_dataset_characteristics(validation)
    if not validation["validation_passed"].eq(True).fillna(False).all():
        raise ValueError("The saved validation report contains an unsuccessful or missing result.")

    characteristics = tables["dataset_characteristics"]
    if characteristics["characteristic"].duplicated().any():
        raise ValueError("The saved characteristics report contains duplicate metric names.")
    metrics = characteristics.set_index("characteristic")["actual_value"]
    metadata = tables["metadata_clean"]
    physical = tables["physical_image_inventory"]
    if metadata.empty:
        raise ValueError("The saved clean metadata is empty.")
    observed = pd.Series({
        "metadata_annotation_rows_clean": len(metadata),
        "unique_metadata_frames": metadata["image_key"].nunique(),
        "unique_labelled_video_frames": len(metadata[FRAME_COLUMNS].drop_duplicates()),
        "finding_classes": metadata[CLASS_COLUMN].nunique(),
        "labelled_videos": metadata["video_key"].nunique(),
        "physical_labelled_image_instances": len(physical),
        "unique_labelled_frames": physical["_audit_filename"].nunique(),
        "duplicate_annotation_rows_in_groups": len(tables["duplicate_annotations"]),
        "multi_class_frames": len(tables["multi_class_frames"]),
        "multi_annotation_same_class_pairs": len(tables["multi_annotation_same_class_groups"]),
    }, dtype="int64")
    differences = pd.DataFrame({"loaded_value": observed, "saved_value": metrics.reindex(observed.index)})
    differences = differences.loc[~differences["loaded_value"].eq(differences["saved_value"]).fillna(False)]
    if not differences.empty:
        raise ValueError(f"Saved tables disagree with their report: {differences.to_dict(orient='index')}")
    return tables


def load_legacy_parquet_audit(output_directory):
    """Read the previous all-Parquet export for one-time format conversion."""
    directory = Path(output_directory)
    names = [*AUDIT_STORAGE_SPEC["artifact"], "parquet_export_report"]
    paths = {name: directory / f"{name}.parquet" for name in names}
    missing = [path.name for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing Parquet files: {missing}")

    catalog = pd.read_parquet(paths["parquet_export_report"], engine="pyarrow")
    required = {"artifact", "rows", "columns", "file_size_bytes"}
    if not required.issubset(catalog.columns) or catalog["artifact"].duplicated().any():
        raise ValueError("The saved Parquet catalog is incomplete or ambiguous.")
    if set(catalog["artifact"]) != set(AUDIT_STORAGE_SPEC["artifact"]):
        raise ValueError("The saved Parquet catalog does not describe the required artifact set.")
    catalog_by_name = catalog.set_index("artifact")
    tables = {}
    for name in AUDIT_STORAGE_SPEC["artifact"]:
        path = paths[name]
        record = catalog_by_name.loc[name]
        if path.stat().st_size != record["file_size_bytes"]:
            raise ValueError(f"Saved file size does not match the catalog: {name}")
        # Older exports have no checksum column and remain supported.
        if "file_sha256" in catalog.columns:
            checksum = hashlib.sha256(path.read_bytes()).hexdigest()
            if not isinstance(record["file_sha256"], str) or checksum != record["file_sha256"]:
                raise ValueError(f"Saved file checksum does not match the catalog: {name}")
        table = pd.read_parquet(path, engine="pyarrow")
        if len(table) != record["rows"] or len(table.columns) != record["columns"]:
            raise ValueError(f"Saved table dimensions do not match the catalog: {name}")
        tables[name] = table

    validate_saved_audit_tables(tables)
    # Resolve current paths from DIRS; old absolute paths in a catalog may have moved.
    catalog = catalog.assign(file_path=lambda data: data["artifact"].map(lambda name: str(paths[name])))
    return saved_tables_to_audit(tables), catalog


def audit_json_value(value):
    """Normalize JSON values without rounding floats or stringifying nulls/lists."""
    if isinstance(value, dict):
        return {key: audit_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [audit_json_value(item) for item in value]
    if isinstance(value, np.generic):
        return audit_json_value(value.item())
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return value


def audit_json_text(value):
    return json.dumps(audit_json_value(value), ensure_ascii=False, indent=2, allow_nan=False) + "\n"


def read_audit_table(path, record):
    """Restore the declared format; CSV string keys never undergo type guessing."""
    if record["format"] == "parquet":
        return pd.read_parquet(path, engine="pyarrow")
    if record["format"] == "json":
        return pd.read_json(path, orient="table", precise_float=True)
    if record["format"] == "csv":
        table = pd.read_csv(
            path, dtype=record["dtypes"], keep_default_na=False,
            float_precision="round_trip", encoding="utf-8",
        )
        for column in record["json_columns"]:
            table[column] = table[column].map(json.loads)
        return table
    raise ValueError(f"Unsupported saved format: {record['format']}")


def load_saved_dataset_audit(output_directory):
    """Check and load Parquet metadata plus CSV/JSON reports without raw inputs."""
    directory = Path(output_directory)
    payload = json.loads((directory / AUDIT_CATALOG_FILENAME).read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or payload.get("format_version") != 1:
        raise ValueError("Invalid or unsupported audit catalog.")
    catalog = pd.DataFrame.from_records(payload["artifacts"])
    required = {"artifact", "format", "filename", "rows", "columns", "file_size_bytes",
                "file_sha256", "dtypes", "json_columns"}
    if not required.issubset(catalog.columns) or catalog["artifact"].duplicated().any():
        raise ValueError("The saved audit catalog is incomplete or ambiguous.")
    if set(catalog["artifact"]) != set(AUDIT_STORAGE_SPEC["artifact"]):
        raise ValueError("The saved audit catalog does not describe the required artifact set.")

    records = catalog.set_index("artifact").to_dict(orient="index")
    tables = {}
    for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
        record = records[spec.artifact]
        filename = f"{spec.artifact}.{spec.format}"
        if (record["filename"] != filename or record["format"] != spec.format
                or record["json_columns"] != list(spec.json_columns)):
            raise ValueError(f"Saved format does not match the specification: {spec.artifact}")
        path = directory / filename
        if path.stat().st_size != record["file_size_bytes"]:
            raise ValueError(f"Saved file size does not match the catalog: {filename}")
        if hashlib.sha256(path.read_bytes()).hexdigest() != record["file_sha256"]:
            raise ValueError(f"Saved file checksum does not match the catalog: {filename}")
        table = read_audit_table(path, record)
        if (table.shape != (record["rows"], record["columns"])
                or list(table.columns) != list(record["dtypes"])):
            raise ValueError(f"Saved table dimensions or columns do not match: {filename}")
        tables[spec.artifact] = table

    validate_saved_audit_tables(tables)
    return saved_tables_to_audit(tables), catalog


def save_dataset_audit(audit, output_directory, preserve_metadata_file=False):
    """Stage and verify all formats before replacing files; write the catalog last.

    Existing all-Parquet exports can be converted without rewriting metadata.
    Only the previously named report Parquets are removed after the new set
    has been reloaded and validated successfully.
    """
    tables = validate_saved_audit_tables(audit_to_saved_tables(audit))
    directory = Path(output_directory)
    directory.mkdir(parents=True, exist_ok=True)
    staged, records = [], []
    catalog_path = directory / AUDIT_CATALOG_FILENAME
    catalog_temporary = catalog_path.with_suffix(".tmp.json")

    try:
        for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
            table = tables[spec.artifact]
            path = directory / f"{spec.artifact}.{spec.format}"
            record = {
                "artifact": spec.artifact, "format": spec.format, "filename": path.name,
                "rows": len(table), "columns": len(table.columns),
                "dtypes": table.dtypes.astype(str).to_dict(),
                "json_columns": list(spec.json_columns),
            }
            if preserve_metadata_file and spec.artifact == "metadata_clean":
                saved_path = path
            else:
                saved_path = path.with_suffix(f".tmp.{spec.format}")
                staged.append((saved_path, path))
                if spec.format == "parquet":
                    table.to_parquet(saved_path, engine="pyarrow", compression="snappy", index=False)
                elif spec.format == "json":
                    payload = {"schema": pd.io.json.build_table_schema(table, index=False),
                               "data": table.to_dict(orient="records")}
                    saved_path.write_text(audit_json_text(payload), encoding="utf-8")
                elif spec.format == "csv":
                    encoded = table.copy()
                    for column in spec.json_columns:
                        encoded[column] = encoded[column].map(
                            lambda value: json.dumps(audit_json_value(value), ensure_ascii=False, allow_nan=False)
                        )
                        record["dtypes"][column] = "object"
                    encoded.to_csv(saved_path, index=False, encoding="utf-8", lineterminator="\n")
                else:
                    raise ValueError(f"Unsupported storage format: {spec.format}")

            restored = read_audit_table(saved_path, record)
            if spec.format == "parquet":
                pd.testing.assert_frame_equal(table.reset_index(drop=True), restored, check_exact=True)
            elif (list(restored.columns) != list(table.columns)
                  or audit_json_value(restored.to_dict(orient="records"))
                  != audit_json_value(table.to_dict(orient="records"))):
                raise ValueError(f"Saved values changed during readback: {spec.artifact}")
            records.append({**record, "file_size_bytes": saved_path.stat().st_size,
                            "file_sha256": hashlib.sha256(saved_path.read_bytes()).hexdigest()})

        payload = {"format_version": 1, "artifacts": records}
        catalog_temporary.write_text(audit_json_text(payload), encoding="utf-8")
        if json.loads(catalog_temporary.read_text(encoding="utf-8")) != payload:
            raise ValueError("Audit catalog readback failed.")
        # No old catalog should certify an interrupted replacement of table files.
        catalog_path.unlink(missing_ok=True)
        for temporary, destination in staged:
            temporary.replace(destination)
        catalog_temporary.replace(catalog_path)
        _, catalog = load_saved_dataset_audit(directory)

        # Retire only superseded reports, after their replacement values are verified.
        for spec in AUDIT_STORAGE_SPEC.itertuples(index=False):
            if spec.format != "parquet":
                (directory / f"{spec.artifact}.parquet").unlink(missing_ok=True)
        (directory / "parquet_export_report.parquet").unlink(missing_ok=True)
        return catalog
    finally:
        for temporary, _ in staged:
            temporary.unlink(missing_ok=True)
        catalog_temporary.unlink(missing_ok=True)


def get_or_build_dataset_audit(output_directory, build_function, force_rebuild=False):
    """Reuse complete saved outputs, or lazily build and save them when needed.

    build_function is called ONLY on a cache miss, an invalid saved set, or an
    explicit rebuild. A cache hit does not scan raw files, call the key builder,
    read raw metadata, or rewrite saved files. Use force_rebuild=True after
    changing the raw dataset or processing rules; source freshness is not inferred.
    """
    reason = "force_rebuild=True"
    if not force_rebuild:
        try:
            directory = Path(output_directory)
            legacy = (not (directory / AUDIT_CATALOG_FILENAME).is_file()
                      and (directory / "parquet_export_report.parquet").is_file())
            loader = load_legacy_parquet_audit if legacy else load_saved_dataset_audit
            audit, catalog = loader(directory)
        except (ImportError, PermissionError):
            raise
        except (OSError, ValueError, KeyError, TypeError) as error:
            reason = f"{type(error).__name__}: {error}"
        else:
            if legacy:
                save_dataset_audit(audit, directory, preserve_metadata_file=True)
                audit, catalog = load_saved_dataset_audit(directory)
                return {"status": "converted_saved_formats", "reason": "legacy_parquet_export",
                        "audit": audit, "export_report": catalog}
            return {"status": "loaded_from_files", "reason": "complete_saved_set",
                    "audit": audit, "export_report": catalog}

    print(f"Rebuild required: {reason}")
    audit = build_function()
    catalog = save_dataset_audit(audit, output_directory)
    # Both branches expose the same persisted tables and audit structure.
    return {"status": "rebuilt_and_saved", "reason": reason,
            "audit": saved_tables_to_audit(audit_to_saved_tables(audit)), "export_report": catalog}


# ------------------------------------------------------------------
# Automatic reuse gate. This REPLACES the old unconditional audit call and
# the separate export cell. Only mounted Drive and DIRS are required
# for a cache hit; df and add_matching_keys are resolved only on rebuilding.
# ------------------------------------------------------------------
def build_current_dataset_audit():
    """Use the original normalized metadata only when a rebuild is required."""
    missing_inputs = [name for name in ("df", "add_matching_keys") if name not in globals()]
    if missing_inputs:
        raise RuntimeError(
            "No reusable saved audit set is available. Run the raw-metadata loading "
            "and normalization cells, then rerun this cell. "
            f"Missing rebuild inputs: {missing_inputs}"
        )
    return audit_dataset(
        dataframe=df,
        dataset_root=DIRS["raw_data_dir"],
        matching_key_function=add_matching_keys,
    )


dataset_audit_result = get_or_build_dataset_audit(
    output_directory=Path(DIRS["curated_data_dir"]) / "dataset_audit",
    build_function=build_current_dataset_audit,
    force_rebuild=False,
)

dataset_audit = dataset_audit_result["audit"]
dataset_audit_status = dataset_audit_result["status"]
df_clean = dataset_audit["dataframe_clean"]
dataset_characteristics_report = dataset_audit["characteristics"]
dataset_validation_report = dataset_audit["validation"]
dataset_diagnostics = dataset_audit["diagnostics"]
physical_image_inventory = dataset_audit["physical_image_inventory"]
audit_export_report = dataset_audit_result["export_report"]

print(f"Stage status: {dataset_audit_status}")
if dataset_audit_status == "loaded_from_files":
    print("Saved audit results loaded and checked. Raw dataset audit was skipped.")
elif dataset_audit_status == "converted_saved_formats":
    print("Saved audit converted to CSV/JSON reports. Clean metadata Parquet was preserved.")
    print("Replacement values verified; superseded report Parquets removed. Raw audit was skipped.")
else:
    print("Raw dataset audit completed. Full outputs were saved and reopened successfully.")

for name, table in audit_to_saved_tables(dataset_audit).items():
    print(f"\n{name}: {len(table):,} rows; showing {min(len(table), DISPLAY_MAX_ROWS):,}")
    with pd.option_context("display.max_rows", DISPLAY_MAX_ROWS, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(table.head(DISPLAY_MAX_ROWS))

print("\nSaved artifacts")
display(audit_export_report.loc[:, [
    "artifact", "format", "filename", "rows", "columns", "file_size_bytes",
]].head(DISPLAY_MAX_ROWS))
print(f"\nPASS: audit outputs available; status={dataset_audit_status}.")
print(f"df_clean contains {len(df_clean):,} annotations; use it in subsequent cells.")
